In [1]:
import os
os.environ['JAVA_HOME'] = r'C:\Program Files\Eclipse Adoptium\jdk-21.0.3.9-hotspot'

In [2]:
import os, re, warnings
import pandas as pd
import numpy as np
import duckdb
warnings.filterwarnings('ignore')

RAW_DIR   = 'data/raw'
CLEAN_DIR = 'data/clean'
DB_PATH   = 'data/warehouse.duckdb'
os.makedirs(CLEAN_DIR, exist_ok=True)

In [3]:
raw_customers   = pd.read_csv(f'{RAW_DIR}/customers.csv')
raw_products    = pd.read_csv(f'{RAW_DIR}/products.csv')
raw_orders      = pd.read_csv(f'{RAW_DIR}/orders.csv')
raw_order_items = pd.read_csv(f'{RAW_DIR}/order_items.csv')

for name, df in [('customers', raw_customers), ('products', raw_products),
                 ('orders', raw_orders), ('order_items', raw_order_items)]:
    print(f'{name:15s} {df.shape}   nulls: {df.isnull().sum().to_dict()}')

customers       (357, 4)   nulls: {'customer_id': 1, 'email': 2, 'country': 0, 'created_at': 0}
products        (155, 4)   nulls: {'product_id': 0, 'name': 1, 'category': 1, 'price': 0}
orders          (456, 4)   nulls: {'order_id': 0, 'customer_id': 1, 'order_status': 0, 'created_at': 0}
order_items     (914, 4)   nulls: {'order_item_id': 0, 'order_id': 0, 'product_id': 0, 'quantity': 0}


In [4]:
EMAIL_RE = r'^[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}$'

def clean_customers(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.dropna(subset=['customer_id'])
    df = df.drop_duplicates(subset=['customer_id'], keep='first')
    df['customer_id'] = df['customer_id'].astype(int)
    df['is_email_valid'] = df['email'].str.match(EMAIL_RE, na=False)
    df.loc[~df['is_email_valid'], 'email'] = np.nan
    df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
    return df.reset_index(drop=True)

clean_cust = clean_customers(raw_customers)
print(f'customers: {len(raw_customers)} → {len(clean_cust)} rows')
clean_cust.head()

customers: 357 → 353 rows


,customer_id,email,country,created_at,is_email_valid
0,1,user1@example.com,DE,2024-01-04 23:17:00,True
1,2,user2@example.com,PL,2024-01-29 04:47:00,True
2,3,user3@example.com,DE,2024-03-27 23:57:00,True
3,4,user4@example.com,CZ,2024-01-12 18:27:00,True
4,5,user5@example.com,US,2024-01-04 02:13:00,True


In [5]:
def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.dropna(subset=['product_id'])
    df = df.drop_duplicates(subset=['product_id'], keep='first')
    df = df[df['price'] > 0]
    df['product_id'] = df['product_id'].astype(int)
    df['price'] = df['price'].astype(float)
    return df.reset_index(drop=True)

clean_prod = clean_products(raw_products)
print(f'products: {len(raw_products)} → {len(clean_prod)} rows')
clean_prod.head()

products: 155 → 152 rows


,product_id,name,category,price
0,1001,Toys Product 1001,Toys,810.03
1,1002,Stationery Product 1002,Stationery,797.22
2,1003,Stationery Product 1003,Stationery,1246.05
3,1004,Stationery Product 1004,Stationery,1371.25
4,1005,Books Product 1005,Books,1451.14


In [6]:
VALID_STATUSES = {'pending', 'completed', 'cancelled', 'returned', 'shipped'}

def clean_orders(df: pd.DataFrame, valid_customers: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.drop_duplicates(subset=['order_id'], keep='first')
    df['order_status'] = df['order_status'].str.lower().str.strip()
    df = df[df['order_status'].isin(VALID_STATUSES)]
    df = df.dropna(subset=['customer_id'])
    df['customer_id'] = df['customer_id'].astype(int)
    df = df[df['customer_id'].isin(valid_customers['customer_id'])]
    df['created_at'] = pd.to_datetime(df['created_at'], errors='coerce')
    return df.reset_index(drop=True)

clean_ord = clean_orders(raw_orders, clean_cust)
print(f'orders: {len(raw_orders)} → {len(clean_ord)} rows')
print('statuses after clean:', clean_ord['order_status'].value_counts().to_dict())
clean_ord.head()

orders: 456 → 453 rows
statuses after clean: {'completed': 327, 'cancelled': 65, 'pending': 60, 'returned': 1}


,order_id,customer_id,order_status,created_at
0,5001,307,completed,2024-05-09 06:49:00
1,5002,71,completed,2024-04-16 00:31:00
2,5003,221,completed,2024-06-08 22:14:00
3,5004,257,pending,2024-07-13 11:04:00
4,5005,204,completed,2024-04-03 14:58:00


In [7]:
def clean_order_items(df: pd.DataFrame, valid_orders: pd.DataFrame,
                      valid_products: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df = df.drop_duplicates(subset=['order_item_id'], keep='first')
    df = df[df['quantity'] > 0]
    df['order_id']   = df['order_id'].astype(int)
    df['product_id'] = df['product_id'].astype(int)
    df = df[df['order_id'].isin(valid_orders['order_id'])]
    df = df[df['product_id'].isin(valid_products['product_id'])]
    return df.reset_index(drop=True)

clean_items = clean_order_items(raw_order_items, clean_ord, clean_prod)
print(f'order_items: {len(raw_order_items)} → {len(clean_items)} rows')
clean_items.head()

order_items: 914 → 907 rows


,order_item_id,order_id,product_id,quantity
0,1,5001,1142,5
1,2,5001,1068,3
2,3,5002,1061,1
3,4,5002,1013,4
4,5,5003,1141,4


In [8]:
clean_cust.to_csv(f'{CLEAN_DIR}/customers.csv', index=False)
clean_prod.to_csv(f'{CLEAN_DIR}/products.csv', index=False)
clean_ord.to_csv(f'{CLEAN_DIR}/orders.csv', index=False)
clean_items.to_csv(f'{CLEAN_DIR}/order_items.csv', index=False)
print('Clean CSVs saved to', CLEAN_DIR)

Clean CSVs saved to data/clean


In [9]:
con = duckdb.connect(DB_PATH)

tables = {
    'clean_customers':   clean_cust,
    'clean_products':    clean_prod,
    'clean_orders':      clean_ord,
    'clean_order_items': clean_items,
}

for tbl, df in tables.items():
    con.execute(f'DROP TABLE IF EXISTS {tbl}')
    con.execute(f'CREATE TABLE {tbl} AS SELECT * FROM df')
    n = con.execute(f'SELECT COUNT(*) FROM {tbl}').fetchone()[0]
    print(f'  {tbl}: {n} rows')

  clean_customers: 353 rows
  clean_products: 152 rows
  clean_orders: 453 rows
  clean_order_items: 907 rows


In [10]:
marts = {
    'mart_revenue_by_product': """
        CREATE OR REPLACE TABLE mart_revenue_by_product AS
        SELECT p.product_id, p.name AS product_name, p.category,
               SUM(oi.quantity)           AS total_units_sold,
               SUM(oi.quantity * p.price) AS total_revenue,
               COUNT(DISTINCT oi.order_id) AS order_count
        FROM clean_order_items oi
        JOIN clean_products p ON oi.product_id = p.product_id
        GROUP BY p.product_id, p.name, p.category
        ORDER BY total_revenue DESC
    """,
    'mart_revenue_by_customer': """
        CREATE OR REPLACE TABLE mart_revenue_by_customer AS
        SELECT c.customer_id, c.email, c.country,
               COUNT(DISTINCT o.order_id)   AS total_orders,
               SUM(oi.quantity * p.price)   AS total_spent,
               MIN(o.created_at)            AS first_order_at,
               MAX(o.created_at)            AS last_order_at
        FROM clean_customers c
        JOIN clean_orders o      ON c.customer_id = o.customer_id
        JOIN clean_order_items oi ON o.order_id   = oi.order_id
        JOIN clean_products p    ON oi.product_id = p.product_id
        GROUP BY c.customer_id, c.email, c.country
        ORDER BY total_spent DESC
    """,
    'mart_monthly_sales': """
        CREATE OR REPLACE TABLE mart_monthly_sales AS
        SELECT DATE_TRUNC('month', o.created_at) AS month,
               COUNT(DISTINCT o.order_id)         AS orders,
               COUNT(DISTINCT o.customer_id)      AS unique_customers,
               SUM(oi.quantity)                   AS units_sold,
               ROUND(SUM(oi.quantity * p.price), 2) AS revenue
        FROM clean_orders o
        JOIN clean_order_items oi ON o.order_id   = oi.order_id
        JOIN clean_products p    ON oi.product_id = p.product_id
        WHERE o.order_status = 'completed'
        GROUP BY 1 ORDER BY 1
    """,
    'mart_order_status_summary': """
        CREATE OR REPLACE TABLE mart_order_status_summary AS
        SELECT order_status,
               COUNT(*)                    AS order_count,
               COUNT(DISTINCT customer_id) AS unique_customers
        FROM clean_orders
        GROUP BY order_status ORDER BY order_count DESC
    """,
    'mart_category_performance': """
        CREATE OR REPLACE TABLE mart_category_performance AS
        SELECT p.category,
               COUNT(DISTINCT p.product_id)      AS product_count,
               SUM(oi.quantity)                  AS units_sold,
               ROUND(SUM(oi.quantity * p.price), 2) AS total_revenue,
               ROUND(AVG(p.price), 2)            AS avg_price
        FROM clean_order_items oi
        JOIN clean_products p ON oi.product_id = p.product_id
        GROUP BY p.category ORDER BY total_revenue DESC
    """
}

for name, sql in marts.items():
    con.execute(sql)
    n = con.execute(f'SELECT COUNT(*) FROM {name}').fetchone()[0]
    print(f'  {name}: {n} rows')

  mart_revenue_by_product: 149 rows
  mart_revenue_by_customer: 245 rows
  mart_monthly_sales: 5 rows
  mart_order_status_summary: 4 rows
  mart_category_performance: 7 rows


In [11]:
print('=== Revenue by category ===')
display(con.execute('SELECT * FROM mart_category_performance').df())

print('\n=== Monthly sales ===')
display(con.execute('SELECT * FROM mart_monthly_sales').df())

print('\n=== Order status summary ===')
display(con.execute('SELECT * FROM mart_order_status_summary').df())

print('\n=== Top 5 customers by spend ===')
display(con.execute('SELECT * FROM mart_revenue_by_customer LIMIT 5').df())

print('\n=== Top 5 products by revenue ===')
display(con.execute('SELECT * FROM mart_revenue_by_product LIMIT 5').df())

=== Revenue by category ===


,category,product_count,units_sold,total_revenue,avg_price
0,Books,27,522.0,413146.07,822.54
1,Toys,25,484.0,346362.27,734.72
2,Stationery,23,415.0,287763.50,692.17
3,Beauty,21,377.0,240287.06,650.25
4,Furniture,17,301.0,230897.77,763.22
5,Sports,22,430.0,201478.05,487.89
6,Electronics,14,238.0,122969.43,528.09



=== Monthly sales ===


,month,orders,unique_customers,units_sold,revenue
0,2024-04-01,76,66,486.0,344948.59
1,2024-05-01,99,79,645.0,383624.43
2,2024-06-01,87,74,505.0,323191.82
3,2024-07-01,63,58,393.0,265728.02
4,NaT,1,1,1.0,1206.74



=== Order status summary ===


,order_status,order_count,unique_customers
0,completed,327,207
1,cancelled,65,61
2,pending,60,55
3,returned,1,1



=== Top 5 customers by spend ===


,customer_id,email,country,total_orders,total_spent,first_order_at,last_order_at
0,44,user44@example.com,UA,3,30203.22,2024-04-06 07:45:00,2024-04-18 20:52:00
1,10,user10@example.com,UA,3,25054.40,2024-04-20 13:44:00,2024-05-16 15:34:00
2,114,user114@example.com,NL,4,24156.38,2024-04-08 17:37:00,2024-07-29 00:26:00
3,335,user335@example.com,UA,5,22803.45,2024-04-05 14:16:00,2024-06-08 12:33:00
4,204,user204@example.com,ES,4,21745.49,2024-04-03 14:58:00,2024-07-05 09:56:00



=== Top 5 products by revenue ===


,product_id,product_name,category,total_units_sold,total_revenue,order_count
0,1122,Toys Product 1122,Toys,32.0,39296.00,9
1,1031,Furniture Product 1031,Furniture,26.0,34608.86,8
2,1081,Books Product 1081,Books,32.0,34213.76,10
3,1039,Books Product 1039,Books,24.0,33808.80,8
4,1147,Stationery Product 1147,Stationery,30.0,30539.10,10


In [12]:
con.close()
print('DuckDB saved to', DB_PATH)

DuckDB saved to data/warehouse.duckdb


In [13]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName('ETL_Lecture13')
    .config('spark.ui.enabled', 'false')
    .config('spark.sql.session.timeZone', 'UTC')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('ERROR')
print('Spark version:', spark.version)

Spark version: 3.5.8


In [14]:
sp_customers   = spark.read.csv(f'{RAW_DIR}/customers.csv',   header=True, inferSchema=True)
sp_products    = spark.read.csv(f'{RAW_DIR}/products.csv',    header=True, inferSchema=True)
sp_orders      = spark.read.csv(f'{RAW_DIR}/orders.csv',      header=True, inferSchema=True)
sp_order_items = spark.read.csv(f'{RAW_DIR}/order_items.csv', header=True, inferSchema=True)

EMAIL_RE_SPARK = r'^[A-Za-z0-9._%+\-]+@[A-Za-z0-9.\-]+\.[A-Za-z]{2,}$'

sp_cust = (
    sp_customers
    .filter(F.col('customer_id').isNotNull())
    .dropDuplicates(['customer_id'])
    .withColumn('is_email_valid', F.col('email').rlike(EMAIL_RE_SPARK))
    .withColumn('email', F.when(F.col('is_email_valid'), F.col('email')))
    .withColumn('created_at', F.try_to_timestamp('created_at', F.lit('yyyy-MM-dd HH:mm:ss')))
    .withColumn('customer_id', F.col('customer_id').cast('int'))
)

sp_prod = (
    sp_products
    .filter(F.col('product_id').isNotNull())
    .dropDuplicates(['product_id'])
    .filter(F.col('price') > 0)
    .withColumn('product_id', F.col('product_id').cast('int'))
)

sp_ord = (
    sp_orders
    .dropDuplicates(['order_id'])
    .withColumn('order_status', F.lower(F.trim(F.col('order_status'))))
    .filter(F.col('order_status').isin(['pending','completed','cancelled','returned','shipped']))
    .filter(F.col('customer_id').isNotNull())
    .withColumn('customer_id', F.col('customer_id').cast('int'))
    .join(sp_cust.select(F.col('customer_id').alias('_cid')),
          F.col('customer_id') == F.col('_cid'), 'inner')
    .drop('_cid')
    .withColumn('created_at', F.try_to_timestamp('created_at', F.lit('yyyy-MM-dd HH:mm:ss')))
)

sp_items = (
    sp_order_items
    .dropDuplicates(['order_item_id'])
    .filter(F.col('quantity') > 0)
    .withColumn('order_id',   F.col('order_id').cast('int'))
    .withColumn('product_id', F.col('product_id').cast('int'))
    .join(sp_ord.select(F.col('order_id').alias('_oid')),
          F.col('order_id') == F.col('_oid'), 'inner').drop('_oid')
    .join(sp_prod.select(F.col('product_id').alias('_pid')),
          F.col('product_id') == F.col('_pid'), 'inner').drop('_pid')
)

print('customers :', sp_cust.count())
print('products  :', sp_prod.count())
print('orders    :', sp_ord.count())
print('order_items:', sp_items.count())
spark.stop()

customers : 353
products  : 152
orders    : 453
order_items: 907


In [15]:
test_code = '''
import pandas as pd
import numpy as np
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(__file__), ".."))

EMAIL_RE = r"^[A-Za-z0-9._%+\\-]+@[A-Za-z0-9.\\-]+\\.[A-Za-z]{2,}$"
VALID_STATUSES = {"pending", "completed", "cancelled", "returned", "shipped"}

def clean_customers(df):
    df = df.copy()
    df = df.dropna(subset=["customer_id"])
    df = df.drop_duplicates(subset=["customer_id"], keep="first")
    df["customer_id"] = df["customer_id"].astype(int)
    df["is_email_valid"] = df["email"].str.match(EMAIL_RE, na=False)
    df.loc[~df["is_email_valid"], "email"] = np.nan
    df["created_at"] = pd.to_datetime(df["created_at"], errors="coerce")
    return df.reset_index(drop=True)

def clean_products(df):
    df = df.copy()
    df = df.dropna(subset=["product_id"])
    df = df.drop_duplicates(subset=["product_id"], keep="first")
    df = df[df["price"] > 0]
    df["product_id"] = df["product_id"].astype(int)
    return df.reset_index(drop=True)

def clean_orders(df, valid_customers):
    df = df.copy()
    df = df.drop_duplicates(subset=["order_id"], keep="first")
    df["order_status"] = df["order_status"].str.lower().str.strip()
    df = df[df["order_status"].isin(VALID_STATUSES)]
    df = df.dropna(subset=["customer_id"])
    df["customer_id"] = df["customer_id"].astype(int)
    df = df[df["customer_id"].isin(valid_customers["customer_id"])]
    return df.reset_index(drop=True)

def clean_order_items(df, valid_orders, valid_products):
    df = df.copy()
    df = df.drop_duplicates(subset=["order_item_id"], keep="first")
    df = df[df["quantity"] > 0]
    df["order_id"]   = df["order_id"].astype(int)
    df["product_id"] = df["product_id"].astype(int)
    df = df[df["order_id"].isin(valid_orders["order_id"])]
    df = df[df["product_id"].isin(valid_products["product_id"])]
    return df.reset_index(drop=True)

# --- tests ---

def test_customers_drops_null_id():
    df = pd.DataFrame({"customer_id":[None,1],"email":["a@b.com","a@b.com"],
                       "country":["DE","DE"],"created_at":["2024-01-01","2024-01-01"]})
    assert len(clean_customers(df)) == 1

def test_customers_removes_duplicates():
    df = pd.DataFrame({"customer_id":[1,1],"email":["a@b.com","c@d.com"],
                       "country":["DE","PL"],"created_at":["2024-01-01","2024-01-02"]})
    assert len(clean_customers(df)) == 1

def test_customers_nulls_invalid_email():
    df = pd.DataFrame({"customer_id":[1,2],"email":["bad-email","good@ok.com"],
                       "country":["DE","DE"],"created_at":["2024-01-01","2024-01-01"]})
    result = clean_customers(df)
    assert pd.isna(result.loc[result.customer_id==1,"email"].iloc[0])
    assert result.loc[result.customer_id==2,"email"].iloc[0] == "good@ok.com"

def test_products_drops_nonpositive_price():
    df = pd.DataFrame({"product_id":[1,2,3],"name":["A","B","C"],
                       "category":["X","X","X"],"price":[100,-5,0]})
    assert len(clean_products(df)) == 1

def test_orders_normalises_status():
    cust = pd.DataFrame({"customer_id":[1]})
    df = pd.DataFrame({"order_id":[1],"customer_id":[1],
                       "order_status":["COMPLETED"],"created_at":["2024-01-01"]})
    result = clean_orders(df, cust)
    assert result["order_status"].iloc[0] == "completed"

def test_orders_drops_unknown_status():
    cust = pd.DataFrame({"customer_id":[1]})
    df = pd.DataFrame({"order_id":[1,2],"customer_id":[1,1],
                       "order_status":["completed","mystery"],"created_at":["2024-01-01","2024-01-01"]})
    assert len(clean_orders(df, cust)) == 1

def test_orders_drops_missing_customer():
    cust = pd.DataFrame({"customer_id":[1]})
    df = pd.DataFrame({"order_id":[1,2],"customer_id":[1,999],
                       "order_status":["completed","completed"],"created_at":["2024-01-01","2024-01-01"]})
    assert len(clean_orders(df, cust)) == 1

def test_order_items_drops_nonpositive_qty():
    orders   = pd.DataFrame({"order_id":[1]})
    products = pd.DataFrame({"product_id":[10]})
    df = pd.DataFrame({"order_item_id":[1,2,3],"order_id":[1,1,1],
                       "product_id":[10,10,10],"quantity":[3,0,-1]})
    assert len(clean_order_items(df, orders, products)) == 1

def test_order_items_drops_orphan_order():
    orders   = pd.DataFrame({"order_id":[1]})
    products = pd.DataFrame({"product_id":[10]})
    df = pd.DataFrame({"order_item_id":[1,2],"order_id":[1,999],
                       "product_id":[10,10],"quantity":[1,1]})
    assert len(clean_order_items(df, orders, products)) == 1
'''

os.makedirs('tests', exist_ok=True)
with open('tests/test_cleaner.py', 'w') as f:
    f.write(test_code)
print('test file written')

test file written


In [16]:
import subprocess
result = subprocess.run(
    ['python', '-m', 'pytest', 'tests/test_cleaner.py', '-v', '--tb=short'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)

============================= test session starts =============================
platform win32 -- Python 3.10.2, pytest-9.0.3, pluggy-1.6.0 -- c:\Users\gulya\AppData\Local\Programs\Python\Python310\python.exe
cachedir: .pytest_cache
rootdir: d:\CampHomework\lesson_13HW
collecting ... collected 9 items

tests/test_cleaner.py::test_customers_drops_null_id PASSED               [ 11%]
tests/test_cleaner.py::test_customers_removes_duplicates PASSED          [ 22%]
tests/test_cleaner.py::test_customers_nulls_invalid_email PASSED         [ 33%]
tests/test_cleaner.py::test_products_drops_nonpositive_price PASSED      [ 44%]
tests/test_cleaner.py::test_orders_normalises_status PASSED              [ 55%]
tests/test_cleaner.py::test_orders_drops_unknown_status PASSED           [ 66%]
tests/test_cleaner.py::test_orders_drops_missing_customer PASSED         [ 77%]
tests/test_cleaner.py::test_order_items_drops_nonpositive_qty PASSED     [ 88%]
tests/test_cleaner.py::test_order_items_drops_orphan_ord